# 100 Jobs Timing Benchmark

Submits 100 jobs from DS to DO and times each step of the flow using Jupyter's built-in `%%time` / `%time` cell magics.

### Timed Steps
1. Submit 100 jobs (DS)
2. DO sync
3. DS sync
4. Show jobs (DS)
5. Submit one extra job (DS)
6. DO sync, approve, run, sync results back
7. DS sync to download results
8. Read job output (DS)

### Prerequisites
- Same as `test_manual_e2e.ipynb`
- A `.env` file with: `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `TOKEN_DS`

---

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]

do_token_path = Path(os.environ["TOKEN_DO"])
ds_token_path = Path(os.environ["TOKEN_DS"])

assert do_token_path.exists() and ds_token_path.exists()

### 1.1 Login

In [ ]:
import syft_client as sc

In [ ]:
do_client = sc.login_do(email=DO_EMAIL, token_path=str(do_token_path))

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(ds_token_path))

### 1.2 Optional — Wipe prior state

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

### 1.3 Establish peer connection

In [ ]:
ds_client.add_peer(DO_EMAIL)

do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

---

## 2. Prepare a reusable job script

The same script is reused for all 100 jobs — only the job name differs.

In [ ]:
import tempfile

JOB_CODE = '''import os, json

result = {"message": "hello from benchmark job"}
os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f)
print(result)
'''

_job_dir = Path(tempfile.mkdtemp(prefix="syft_bench_"))
BENCH_SCRIPT = _job_dir / "main.py"
BENCH_SCRIPT.write_text(JOB_CODE)
print(f"Job script: {BENCH_SCRIPT}")

---

## 3. Submit 100 jobs (DS) — timed

In [ ]:
N_JOBS = 100
JOB_PREFIX = "bench_job"

In [ ]:
%%time
for i in range(N_JOBS):
    ds_client.submit_python_job(
        user=DO_EMAIL,
        code_path=str(BENCH_SCRIPT),
        job_name=f"{JOB_PREFIX}_{i:03d}",
        force_submission=True,
    )

---

## 4. DO sync — timed

In [ ]:
%%time
do_client.sync()

In [ ]:
do_client.sync()

---

## 5. DS sync — timed

In [ ]:
%%time
ds_client.sync()

---

## 6. Show jobs (DS) — timed

In [ ]:
%time ds_jobs = ds_client.jobs

In [ ]:
# ds_jobs

In [ ]:
# ds_jobs

---

## 7. Submit + download one more job — timed end to end

Each step is timed individually so you can see where the wall-clock time goes.

In [ ]:
EXTRA_JOB_NAME = "bench_extra_job2"

### 7.1 DS submits the extra job

In [ ]:
%%time
ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(BENCH_SCRIPT),
    job_name=EXTRA_JOB_NAME,
    force_submission=True,
)

### 7.2 DO sync to pick up the extra job

In [ ]:
%%time
do_client.sync()

### 7.3 DO approves and runs the extra job

In [ ]:
%time do_client.jobs[EXTRA_JOB_NAME].approve()

In [ ]:
%%time
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

### 7.4 DO sync to push results back

In [ ]:
%%time
do_client.sync()

### 7.5 DS sync to download the result

In [ ]:
%%time
ds_client.sync()

### 7.6 DS reads the output

In [ ]:
%%time
ds_extra_job = ds_client.jobs[EXTRA_JOB_NAME]
print(f"Status: {ds_extra_job.status}")
print(f"Output files: {ds_extra_job.output_paths}")
if ds_extra_job.output_paths:
    print(f"Result: {ds_extra_job.output_paths[0].read_text()}")